# 02 · GAN / DCGAN MNIST —— 造假钞 vs 验钞机

**家族位置**：07 生成式模型第 2 站。01 的 AE/VAE 是自己检查重建，GAN 改成生成器 G 造假、判别器 D 验真；本章 MLP-GAN 与 DCGAN 同 MNIST 小集合同台。

**学习目标**：理解 minimax/非饱和目标；看 MLP 展平生成 vs DCGAN 卷积生成；用固定噪声快照与多样性 proxy 观察训练、模式坍塌。

## 1. 原理：两个学生互卷

### 通俗理解

**一句话**：G 像印假钞，D 像验钞机；G 越会造，D 越会验，互相升级后 G 才能造出以假乱真的数字。

MLP-GAN 把 28×28 摊平再生成，像把拼图全揉成一条纸；DCGAN 保留二维空间，用卷积/反卷积知道“相邻像素应该相邻”，通常图更稳。

### 结构账

```
z~N(0,I) → G(z)=fake；D(real)=1，D(fake)=0
D loss = BCE(D(real),1)+BCE(D(G(z)),0)
G loss = BCE(D(G(z)),1)（非饱和目标，避免早期梯度太小）
```

模式坍塌：G 只会印一种“最容易骗过 D”的数字，loss 可能正常但样本多样性消失。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
assert (ROOT/'common').exists()
sys.path.insert(0,str(ROOT))
from common.data import load_mnist_local
from common.models import GeneratorMLP,DiscriminatorMLP,GeneratorDCGAN,DiscriminatorDCGAN
from common.engine import fit_gan
from common.utils import set_seed,setup_chinese_font,count_params,show_mnist
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
Xtr,ytr,Xva,yva,Xte,yte=load_mnist_local(6000,1000,seed=0)
loader=DataLoader(TensorDataset((Xtr*0.3081+0.1307).clamp(0,1),ytr),batch_size=128,shuffle=True)
print(f'train {tuple(Xtr.shape)} | MNIST [0,1] GAN input | test {tuple(Xte.shape)}')
# fig0: real data
fig,ax=plt.subplots(2,8,figsize=(9,2.8))
for a,x in zip(ax.flat, Xtr[:16]): a.imshow(show_mnist(x),cmap='gray'); a.axis('off')
plt.suptitle('MNIST 真图：D 要学会识别的样本'); plt.tight_layout(); plt.savefig(FIGS/'fig0_real.png',dpi=150,bbox_inches='tight'); plt.show()

## 2. 数据与训练：同 z、同预算、同非饱和目标

In [ ]:
Z_DIM,EPOCHS,LR=64,20,2e-4
fixed_z=torch.randn(64,Z_DIM)
mlp_g,mlp_d=GeneratorMLP(Z_DIM),DiscriminatorMLP()
dc_g,dc_d=GeneratorDCGAN(Z_DIM),DiscriminatorDCGAN()
print(f'MLP G/D params={count_params(mlp_g)}/{count_params(mlp_d)} | DCGAN G/D params={count_params(dc_g)}/{count_params(dc_d)}')
h_mlp=fit_gan(mlp_g,mlp_d,loader,EPOCHS,Z_DIM,LR,seed=0,fixed_z=fixed_z)
h_dc=fit_gan(dc_g,dc_d,loader,EPOCHS,Z_DIM,LR,seed=0,fixed_z=fixed_z)
fig,ax=plt.subplots(1,2,figsize=(9,3.4))
ax[0].plot(h_mlp['g'],label='MLP G'); ax[0].plot(h_mlp['d'],label='MLP D'); ax[0].plot(h_dc['g'],label='DC G',ls='--'); ax[0].plot(h_dc['d'],label='DC D',ls='--'); ax[0].set_title('G/D loss'); ax[0].legend(fontsize=8)
ax[1].plot(h_mlp['real_score'],label='MLP real'); ax[1].plot(h_mlp['fake_score'],label='MLP fake'); ax[1].plot(h_dc['real_score'],label='DC real',ls='--'); ax[1].plot(h_dc['fake_score'],label='DC fake',ls='--'); ax[1].set_ylim(0,1); ax[1].set_title('D score'); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(FIGS/'fig1_gan_loss.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 固定噪声：训练过程中的生成变化

In [ ]:
def grid(samples,title,path):
    fig,ax=plt.subplots(2,8,figsize=(9,2.8))
    for a,x in zip(ax.flat,samples[:16]): a.imshow(x[0],cmap='gray',vmin=0,vmax=1); a.axis('off')
    plt.suptitle(title); plt.tight_layout(); plt.savefig(path,dpi=150,bbox_inches='tight'); plt.show()
with torch.no_grad():
    sample_m=mlp_g(fixed_z).cpu(); sample_d=dc_g(fixed_z).cpu()
grid(sample_m,'MLP-GAN 固定 z 末期样本',FIGS/'fig2_gan_sample.png')
fig,ax=plt.subplots(2,4,figsize=(9,4.6))
for j,ep in enumerate([0,1,7,19]):
    for a,x in zip(ax[:,j], [h_mlp['snapshots'][ep][0],h_dc['snapshots'][ep][0]]): a.imshow(x[0],cmap='gray',vmin=0,vmax=1); a.axis('off')
    ax[0,j].set_title(f'ep {ep+1}')
ax[0,0].set_ylabel('MLP'); ax[1,0].set_ylabel('DCGAN'); plt.suptitle('同一 z 的生成轨迹'); plt.tight_layout(); plt.savefig(FIGS/'fig3_progression.png',dpi=150,bbox_inches='tight'); plt.show()

## 4. 多样性：模式坍塌的实用 proxy

In [ ]:
def diversity(samples):
    # 每张图平均像素标准差 + pairwise feature distance 的简易 proxy，不冒充 FID
    x=samples.flatten(1); d=torch.pdist(x).mean().item(); return x.std().item(),d
with torch.no_grad(): m=mlp_g(fixed_z).cpu(); d=dc_g(fixed_z).cpu()
print(f'MLP diversity pixel/pair={diversity(m)} | DCGAN={diversity(d)}')
fig,ax=plt.subplots(1,2,figsize=(9,2.8))
for a,s,t in zip(ax,[m,d],['MLP-GAN samples','DCGAN samples']):
    for x in s[:16]: a.imshow(x[0],cmap='gray',vmin=0,vmax=1); a.axis('off')
    a.set_title(t)
plt.suptitle('最终样本多样性：看整行是否都变成同一张脸'); plt.tight_layout(); plt.savefig(FIGS/'fig4_collapse.png',dpi=150,bbox_inches='tight'); plt.show()

## 5. 总结与下一步

MLP-GAN 与 DCGAN 都完成 G/D 对抗闭环；卷积版保留空间局部性，通常更适合图像。loss 不是质量分数，必须同时看固定 z、训练轨迹和多样性；本章 proxy 不冒充 FID。下一步 `03_Diffusion_MNIST`：从逐步加噪到逐步去噪。